# Notebook 2 -- How a single market evolves over time

Notebook 1 worked out what each column means and which ones matter. This notebook
picks up from that conclusion and asks a different question: **for one market, how
do the orderbook and the trade tape move together over time?**

It is deliberately self-contained -- it re-reads the parquet files and rebuilds the
derived columns, so notebook 1 does not have to be run first.

Key fact carried over from notebook 1: every books row -- `snapshot` **and** `update` --
contains a full top-5 ladder on both sides. So a row is a complete photo of the book,
not a delta. Nothing in the data labels *why* the book changed, so the cause
(someone added, someone cancelled, or someone traded) has to be inferred by comparing
consecutive photos and lining the trades up against them.

### Setup: libraries, raw data, derived columns

In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from utils_for_eda import get_market, show_window

In [2]:
df_trades = pd.read_parquet("../data/trades_823753_pregame.parquet")
df_books  = pd.read_parquet("../data/orderbook_data_823753_pregame.parquet")

print("books ", df_books.shape)
print("trades", df_trades.shape)

books  (58275, 18)
trades (2232, 18)


In [3]:
# Convert the exchange and received timestamps from object/string to datetime, with time zone awareness
for df in [df_trades, df_books]:
    for col in ["recv_ts_utc", "exchange_ts_utc"]:
        # format="ISO8601" is required: exchange_ts_utc is ms-quantized, so stamps
        # that land on a whole second carry no fractional part. pandas guesses one
        # format from the first element and then fails on the rest.
        df[col] = pd.to_datetime(df[col], utc = True, format = "ISO8601")

In [4]:
# Add a column at the 1-second granularity, because millisecond and microsecond makes it hard to analyze the event
# Note: floor(), not round() -- floor gives clean bins that line up with resample/Grouper,
# whereas rounding shifts events across bin boundaries at the half-second.
for df in (df_books, df_trades):
    for ts in ["recv_ts_utc", "exchange_ts_utc"]:
        df[ts.replace("_utc", "_sec")] = df[ts].dt.floor("s")

In [5]:
# Latency between the exchange timestamp and the time we received the message
for df in (df_trades, df_books):
    df["latency_s"] = (df["recv_ts_utc"] - df["exchange_ts_utc"]).dt.total_seconds()

### Slice down to the columns that matter

Same verdict as notebook 1, with one deliberate difference: **the raw microsecond
timestamps come back.** Notebook 1 worked at 1-second granularity, which was right for
aggregate EDA. Here a single market can take dozens of updates inside one second, so
`_sec` would scramble the order of exactly the events we are trying to watch.

Both raw clocks are kept, but **sequencing is done on `recv_ts_utc` throughout**, for one
reason: `exchange_ts_utc` is quantized to whole milliseconds and so cannot separate events
that land in the same millisecond. Across all 33 markets, 6,606 of 58,275 book rows
(11.3%) share an exchange timestamp with another row; `recv_ts_utc` has zero duplicates.
The 757-contract trade below is the case in point -- the book update and the trade it
caused carry the *identical* exchange timestamp, so only the receive clock can tell us
which came first.

Worth recording what is *not* wrong with the exchange clock: sorted by `recv_ts_utc`,
`exchange_ts_utc` never runs backwards -- 0 inversions across all 58,275 rows, snapshots
included. Snapshots do restate stale times (median latency 6.6s vs 0.023s for updates),
which distorts the latency histogram, but it does not reorder events within a market.
So exchange time is the better clock for measuring *elapsed time* between events, since
it is free of network jitter; receive time is the better clock for *ordering* them.

In [6]:
# Dropped everywhere: river_id (1:1 with native_id), game_pk (single game),
#   player_id / player_name (all NaN), market_type (only ever "Over", and only
#   populated for team_totals), collector_run_id (constant), date / hour (derived
#   from recv_ts_utc).

# Identity of the contract. native_id is the ticker; (stat, line, outcome_name)
# is the same information in human-readable form -- verified 1:1 both ways in notebook 1.
cols_id = ["native_id", "stat", "line", "outcome_name"]

# recv_ts_utc leads: raw microseconds, the only clock with no ties, so it defines
# event order. exchange_ts_utc rides along for cross-checking elapsed time.
cols_time = ["recv_ts_utc", "exchange_ts_utc", "latency_s"]

cols_books = [
    *cols_time,
    "msg_type",                        # update vs snapshot -- must filter on this
    *cols_id,
    "best_bid", "best_ask", "spread",
    "bids", "asks",                    # full top-5 depth, needed for liquidity analysis
]

cols_trades = [
    *cols_time,
    *cols_id,
    "price", "qty",
    "aggressor_buy_flag",              # True = aggressor lifted the ask (bought)
                                       # False = aggressor hit the bid (sold)
    "exchange_trade_id",
]
# msg_type omitted from trades: constant "trade"

books  = df_books[cols_books].copy()
trades = df_trades[cols_trades].copy()
print("books ", books.shape)
print("trades", trades.shape)

books  (58275, 13)
trades (2232, 11)


### Pick a market

`get_market` slices both tables down to one `native_id`, returns copies sorted
oldest-first on the raw `recv_ts_utc`, and keeps each row's position in the full
frame as `orig_idx` so we can always jump back.

In [7]:
# How many trades did each market actually see? Pick one with enough activity to watch.
trades["native_id"].value_counts().head(n = 10)

native_id
KXMLBRFI-26AUG051940PITMIL               1234
KXMLBTOTAL-26AUG051940PITMIL-8            385
KXMLBF5TOTAL-26AUG051940PITMIL-4          107
KXMLBTOTAL-26AUG051940PITMIL-7             71
KXMLBTOTAL-26AUG051940PITMIL-6             71
KXMLBTEAMTOTAL-26AUG051940PITMIL-MIL4      70
KXMLBTOTAL-26AUG051940PITMIL-5             38
KXMLBTOTAL-26AUG051940PITMIL-3             38
KXMLBTOTAL-26AUG051940PITMIL-9             28
KXMLBF5TOTAL-26AUG051940PITMIL-5           24
Name: count, dtype: Int64

In [8]:
MARKET = "KXMLBTOTAL-26AUG051940PITMIL-9"      # P(total runs in the game >= 9)

b, t = get_market(MARKET, books, trades)
print(f"{MARKET}\n  book rows: {len(b):,}   ({b['msg_type'].value_counts().to_dict()})")
print(f"  trades   : {len(t):,}")

KXMLBTOTAL-26AUG051940PITMIL-9
  book rows: 4,370   ({'update': 4209, 'snapshot': 161})
  trades   : 28


In [9]:
# The trades, so we have timestamps to walk through one at a time.
t[["recv_ts_utc", "price", "qty", "aggressor_buy_flag"]].head(n = 10)

,recv_ts_utc,price,qty,aggressor_buy_flag
0,2026-08-05 17:40:37.420654+00:00,0.37,18.56,False
1,2026-08-05 18:05:50.351606+00:00,0.37,13.00,False
2,2026-08-05 18:27:59.421822+00:00,0.37,4.00,False
3,2026-08-05 18:27:59.421842+00:00,0.37,73.36,False
4,2026-08-05 18:47:11.770280+00:00,0.36,12.00,False
5,2026-08-05 20:00:57.547629+00:00,0.36,8.45,False
6,2026-08-05 20:03:30.795219+00:00,0.36,7.00,False
7,2026-08-05 20:30:19.542399+00:00,0.36,76.20,False
8,2026-08-05 21:19:07.295179+00:00,0.36,95.10,False
9,2026-08-05 21:31:17.521298+00:00,0.36,4.00,False


### Walk through the book one window at a time

`show_window(b, t, start, seconds=1.0)` prints every book row and every trade inside a
time window, interleaved in time order. Each book row is rendered as the "whiteboard"
-- asks on top, bids below -- with the change in size at each price level versus the
previous book row.

Useful arguments:

| argument | meaning |
| --- | --- |
| `seconds` | window length; widen it to scan a quiet stretch |
| `full=False` | one compact line per book row instead of the whole ladder |
| `depth` | levels shown per side (5 is all the feed gives us) |
| `max_events` | guard so a busy second doesn't print hundreds of lines |

In [10]:
# A trade of 757 contracts. Watch the size disappear from the bid, then the trade print.
show_window(b, t, "2026-08-05 22:57:56")

2026-08-05 22:57:56.000000 -> 22:57:57.000000   5 book rows, 1 trades
  22:57:56.174813  update    best 0.36 / 0.37   (row 51402)
      ASK  0.41 x    10,969.00  
      ASK  0.40 x    16,384.00  
      ASK  0.39 x    66,914.10  
      ASK  0.38 x    37,716.68  
      ASK  0.37 x     6,175.50  
      ----------------------------------  spread 0.01
      BID  0.36 x    42,774.00  <- -757.00
      BID  0.35 x    67,443.97  
      BID  0.34 x    54,653.00  
      BID  0.33 x     8,966.90  
      BID  0.32 x     9,515.00  

  22:57:56.175308  *** TRADE  SELL     757.00 @ 0.36 ***

  22:57:56.227335  update    best 0.36 / 0.37   (row 51403)
      ASK  0.41 x    10,969.00  
      ASK  0.40 x    16,384.00  
      ASK  0.39 x    66,914.10  
      ASK  0.38 x    47,671.68  <- +9,955.00
      ASK  0.37 x     6,175.50  
      ----------------------------------  spread 0.01
      BID  0.36 x    42,774.00  
      BID  0.35 x    67,443.97  
      BID  0.34 x    54,653.00  
      BID  0.33 x     8,966

In [11]:
# Same stretch, compact, over a wider window -- easier to scan for what changed.
show_window(b, t, "2026-08-05 22:57:50", seconds = 15, full = False)

2026-08-05 22:57:50.000000 -> 22:58:05.000000   7 book rows, 1 trades
  22:57:56.174813  update    best 0.36 / 0.37   (row 51402)   bid 0.36 -757.00
  22:57:56.175308  *** TRADE  SELL     757.00 @ 0.36 ***

  22:57:56.227335  update    best 0.36 / 0.37   (row 51403)   ask 0.38 +9,955.00
  22:57:56.265101  update    best 0.36 / 0.37   (row 51404)   bid 0.36 -1,054.97; bid 0.34 +2,000.00
  22:57:56.292906  update    best 0.36 / 0.37   (row 51405)   ask 0.39 -9,888.00
  22:57:56.363422  update    best 0.36 / 0.37   (row 51406)   ask 0.39 -2,000.00; ask 0.37 +2,000.00
  22:57:58.545477  update    best 0.36 / 0.37   (row 51410)   bid 0.34 +22.54
  22:58:00.757244  update    best 0.36 / 0.37   (row 51415)   bid 0.34 -22.54


### Two things to settle before leaning on this data

**Can we ignore `msg_type` and treat every row as a photo of the book?**
Only if snapshots are current. Their *timestamps* are stale -- median latency 6.6s
versus 0.023s for updates -- so the worry is that a snapshot restates an out-of-date
book and injects a change that never happened. The test below looks for the
signature of that: a snapshot whose ladder differs from the row before it, where the
*next* row then reverts to the old state (an A -> B -> A pattern).

**How much of the book's movement is actually trading?** Every trade should consume
resting size, but adds and cancels move the book too. The second cell measures the
split by matching each trade to the nearest book change in the same market.

In [12]:
# Flag every book row that actually changed the top-5 ladder, and test whether
# snapshots restate a stale book.

def ladder(side):
    """One side of a book row -> a hashable tuple, so two rows can be compared."""
    return tuple((round(float(lvl["price"]), 4), float(lvl["qty"])) for lvl in side)

flags, snap_diff, snap_revert = [], 0, 0
for mkt, g in books.groupby("native_id", sort = False):
    g = g.sort_values("recv_ts_utc")
    lad = [(ladder(bid), ladder(ask)) for bid, ask in zip(g["bids"], g["asks"])]
    # First row of each market has nothing to compare against, so it counts as "no change".
    flags.append(pd.Series([False] + [lad[i] != lad[i-1] for i in range(1, len(lad))],
                           index = g.index))

    is_snap = g["msg_type"].eq("snapshot").to_numpy()
    for i in range(1, len(lad) - 1):
        if is_snap[i] and lad[i] != lad[i-1]:
            snap_diff   += 1
            snap_revert += (lad[i+1] == lad[i-1])      # snapshot was out of sync

books["ladder_changed"] = pd.concat(flags)

n, n_chg = len(books), int(books["ladder_changed"].sum())
snap = books["msg_type"].eq("snapshot")
print(f"book rows total                   {n:,}")
print(f"  changed the top-5 ladder        {n_chg:,}  ({100*n_chg/n:.1f}%)")
print(f"  identical to the row before     {n-n_chg:,}  ({100*(n-n_chg)/n:.1f}%)")
print(f"\nsnapshot rows                     {int(snap.sum()):,}")
print(f"  identical to the row before     {int((snap & ~books['ladder_changed']).sum()):,}")
print(f"\nsnapshots that DID change the book  {snap_diff:,}")
print(f"  ...and the next row reverts       {snap_revert:,}"
      f"  ({100*snap_revert/snap_diff:.1f}%)  <- stale-snapshot rate")

book rows total                   58,275
  changed the top-5 ladder        49,965  (85.7%)
  identical to the row before     8,310  (14.3%)

snapshot rows                     4,756
  identical to the row before     2,449

snapshots that DID change the book  2,302
  ...and the next row reverts       39  (1.7%)  <- stale-snapshot rate


In [13]:
# What share of book changes were caused by a trade, rather than by someone
# adding, pulling, or resizing a quote?
chgs = (books.loc[books["ladder_changed"], ["recv_ts_utc", "native_id"]]
             .assign(book_row = lambda d: d.index)
             .sort_values("recv_ts_utc"))

for tol_ms in (5, 50, 500):
    mg = pd.merge_asof(trades.sort_values("recv_ts_utc"), chgs,
                       on = "recv_ts_utc", by = "native_id",
                       direction = "nearest", tolerance = pd.Timedelta(f"{tol_ms}ms"))
    hit  = mg["book_row"].notna()
    rows = mg.loc[hit, "book_row"].nunique()
    print(f"within +/-{tol_ms:4d} ms:  {int(hit.sum()):,} of {len(trades):,} trades matched"
          f"  ->  {rows:,} distinct book changes  ({100*rows/n_chg:.1f}% of all changes)")

within +/-   5 ms:  1,028 of 2,232 trades matched  ->  857 distinct book changes  (1.7% of all changes)
within +/-  50 ms:  1,132 of 2,232 trades matched  ->  942 distinct book changes  (1.9% of all changes)
within +/- 500 ms:  1,199 of 2,232 trades matched  ->  1,000 distinct book changes  (2.0% of all changes)


### What those two cells settle

**1. Snapshot and update are interchangeable as book state.** Only ~1.7% of the
snapshots that changed the book show the revert pattern, so snapshots are stale in
their *timestamp*, not in their *content*. We can read the ladder off any row and
ignore `msg_type`.

The better filter is `ladder_changed` rather than `msg_type`. About 14% of book rows
are identical to the row before them -- half of those are snapshots restating an
unchanged book, and the rest are updates whose activity happened below level 5, where
this feed cannot see it. Filtering on "did the ladder actually move" expresses the
intent directly and catches both cases.

**2. Trading is a small minority of book movement.** Only ~2% of book changes have a
trade next to them. The other ~98% are quotes being posted, pulled, and resized. So
`books` is the primary dataset for watching a market evolve, and the trade tape is the
rarer, more informative event.

**Open question -- do not lean on trade-to-book matching until this is resolved.**
Only about half the trades match *any* top-5 book change, even at a generous 500ms
tolerance. Two plausible explanations, neither verified yet: a single aggressive order
sweeping several resting orders prints multiple trades against one book update, or the
fill happened at a level outside the top 5 that this feed never shows. This matters
for Q3, which depends on tying an action to its effect on the book.

**Consequence for Q1 (market efficiency).** `books` alone gives quoted spread, depth,
and no-arbitrage violations across strikes -- real evidence, but not the whole answer.
Quotes are cheap to post: the PIT team-total chain is quoted tightly and barely trades
(27 trades across 7 strikes), which is not an efficient market so much as an untested
one. The effective spread -- what a taker actually pays, net of the p*(1-p)*0.07 fee --
needs trade prices. And the sharpest efficiency measure, how fast neighbouring strikes
reprice after a large trade, treats the trade as the information event. Books is the
primary dataset; trades carry the more decisive half of the argument.

### To do

- Step through all of this market's trades and check the disappearing size matches `qty` each time.
- Look at quiet stretches: what does the book do when nobody is trading?
- Repeat on a busy market (`KXMLBRFI-...` has 1,234 trades) and a thin one, and compare.